# ML-10 — Content Action Playbook

This playbook turns the model queue into a human-facing review workflow. The wording stays narrow: the outputs are a ranked decision-support signal, not a guarantee of uplift or a fully automated policy.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The ranked queue is best read as a shortlist for human review. In the exported results, the strongest items combine a declining-with-demand signal, visible traffic, and a model decline-risk score. The top-ranked rows are not a promise that refresh will work; they are the pages that look most worth checking first because the evidence points in the same direction.

The reason codes are useful because they explain the ranking in plain language: the model is highlighting pages that appear to be declining while still receiving enough demand to matter, and that may also be visible enough for a refresh to be worth reviewing.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts' / 'ml_utils.py').exists() and (candidate / 'data' / 'processed' / 'refresh_feature_vector.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current notebook path.')


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

queue = pd.read_csv(ROOT / 'outputs' / 'refresh_queue.csv')
summary = json.loads((ROOT / 'outputs' / 'summary.json').read_text())
model_results = json.loads((ROOT / 'outputs' / 'model_results.json').read_text())

queue = queue.copy()
queue['reason_list'] = queue['final_reason_codes'].fillna('').str.split('|')
queue['reason_count'] = queue['reason_list'].apply(len)

top_queue = queue.sort_values('final_rank').head(15)[[
    'final_rank', 'content_id', 'client_id', 'final_refresh_score', 'best_model_probability', 'suggested_action',
    'final_reason_codes', 'impressions_90d', 'sessions_90d', 'avg_position', 'trend_direction'
]].copy()
top_queue['reason_codes_display'] = top_queue['final_reason_codes'].fillna('').str.replace('|', ', ', regex=False)
top_queue


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This output is intended for a reviewer who is triaging refresh candidates. It is most useful when the reviewer is deciding which pages to inspect first, because the model ranks pages that look like stronger refresh candidates than the rest of the pool. It is not intended for fully automated refresh decisions, because the validation evidence shows that the ranking is directional rather than perfect.

The limits matter. The best model in the exported results is random forest, with precision@50 of 0.68 and average precision of 0.61 on the main evaluation, while the grouped validation check was weaker than the random-split result. That means the queue should be treated as a decision-support tool for the current data snapshot, not as a permanent policy or as proof that a refresh will improve performance.

It also depends on the same feature schema and data window used for training, so it is less reliable if the traffic mix, page mix, or business context changes materially.


In [ ]:
best_model = summary.get('best_model', 'random_forest')
best_metrics = model_results.get(best_model, {})

limits_summary = {
    'rows_scored': int(len(queue)),
    'best_model': best_model,
    'top_score': float(queue['final_refresh_score'].max()),
    'high_confidence_items': int((queue['confidence'] == 'high').sum()),
    'precision_at_50': best_metrics.get('precision_at_50'),
    'average_precision': best_metrics.get('average_precision'),
    'queue_output': str(ROOT / 'outputs' / 'refresh_queue.csv'),
}
limits_summary


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

A person should still verify the basic context before acting. The checklist should include whether the trend is truly declining, whether the page still has enough demand to justify effort, whether the content is still relevant to the audience, and whether there are brand, compliance, or seasonality concerns that the model cannot see.

The no-go list is not a statistical rule so much as a caution. Pages with very weak signal, very low visibility, or missing traffic context should not be auto-acted on just because they surfaced in the queue. They should be reviewed by hand or left in the monitoring bucket until more evidence is available.


In [ ]:
queue = queue.copy()
queue['low_signal'] = (
    (queue['impressions_90d'] < 500) |
    (queue['sessions_90d'] < 10) |
    (queue['avg_position'].fillna(0) <= 0)
)

review_ready = queue.loc[
    (queue['confidence'].eq('high')) & (~queue['low_signal'])
].sort_values('final_refresh_score', ascending=False).head(20)

no_go_candidates = queue.loc[
    queue['low_signal'] & (queue['final_refresh_score'] >= queue['final_refresh_score'].quantile(0.8))
].sort_values('final_refresh_score', ascending=False).head(10)

review_ready[['final_rank', 'content_id', 'final_refresh_score', 'confidence', 'suggested_action', 'impressions_90d', 'sessions_90d', 'avg_position', 'final_reason_codes']]

no_go_candidates[['final_rank', 'content_id', 'final_refresh_score', 'confidence', 'impressions_90d', 'sessions_90d', 'avg_position', 'final_reason_codes']]


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The queue should be monitored as a living decision aid. The most important warnings are a sharp shift in the action mix, a jump in the share of high-confidence items, a large change in the top reason codes, or a weak manual-review hit rate compared with the model's original ranking quality.

A retrain or refresh of the scoring logic is worth considering when the traffic mix changes materially, the top features become less stable, or the queue begins surfacing many pages that reviewers reject for reasons the model does not capture.


In [ ]:
reason_counter = {}
for codes in queue['final_reason_codes'].fillna('').tolist():
    for reason in codes.split('|'):
        if reason:
            reason_counter[reason] = reason_counter.get(reason, 0) + 1

monitoring_summary = {
    'action_mix': dict(queue['suggested_action'].value_counts().to_dict()),
    'confidence_mix': dict(queue['confidence'].value_counts().to_dict()),
    'score_summary': {
        'p50': float(queue['final_refresh_score'].quantile(0.5)),
        'p80': float(queue['final_refresh_score'].quantile(0.8)),
        'max': float(queue['final_refresh_score'].max()),
    },
    'top_reasons': dict(sorted(reason_counter.items(), key=lambda item: item[1], reverse=True)[:8]),
}
monitoring_summary


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The paper should build on the exported queue rather than a hand-copied snapshot. I save a compact queue preview and a summary file so the ranking evidence can be reused in the report and linked back to the same model outputs.


In [ ]:
import shutil

export_dir = ROOT / 'work' / 'outputs'
export_dir.mkdir(parents=True, exist_ok=True)

paper_queue = queue.sort_values('final_rank').head(200)[[
    'final_rank', 'content_id', 'client_id', 'final_refresh_score', 'best_model_probability', 'confidence',
    'suggested_action', 'final_reason_codes', 'impressions_90d', 'sessions_90d', 'avg_position', 'trend_direction'
]].copy()
paper_queue.to_csv(export_dir / 'action_playbook_queue.csv', index=False)

paper_summary = {
    'best_model': summary.get('best_model'),
    'rows_scored': int(len(queue)),
    'top_score': float(queue['final_refresh_score'].max()),
    'high_confidence_items': int((queue['confidence'] == 'high').sum()),
    'action_mix': dict(queue['suggested_action'].value_counts().head(10).to_dict()),
    'top_reasons': dict(sorted(reason_counter.items(), key=lambda item: item[1], reverse=True)[:8]),
}
(export_dir / 'action_playbook_summary.json').write_text(json.dumps(paper_summary, indent=2))

shutil.copyfile(ROOT / 'outputs' / 'refresh_queue.csv', export_dir / 'refresh_queue_for_paper.csv')

export_dir, (export_dir / 'action_playbook_queue.csv').exists(), (export_dir / 'action_playbook_summary.json').exists()


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
